In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from openai import OpenAI
openai_client = OpenAI()

In [51]:
def llm(prompt):
    response = openai_client.responses.create(
        model="gpt-5.4-nano",
        input=prompt
    )
    return response.output_text

In [52]:
llm("Hey Whats up?")

'Hey! Not much—just here and ready to help. 😄  \nHow’s your day going?'

In [9]:
question = "I just discovered the course. Can I join now?"

In [13]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

In [14]:
prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

In [15]:
answer = llm(prompt)
print(answer)

Yes, but if you want to receive a certificate, you need to submit your project while submissions are still open.


In [ ]:
def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)



In [5]:
import requests

In [6]:
docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

In [7]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

In [8]:
len(documents)

1342

In [9]:
documents[2]

{'id': '3f1424af17',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: Can I still join the course after the start date?',
 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}

In [40]:
from minsearch import Index

In [41]:
index = Index(text_fields = ["question","section","answer"],
              keyword_fields = ["course"]
             )
index.fit(documents)

In [42]:
question = "I just discovered the course. Can I join now?"

In [13]:
question = "I just discovered the course. Can I join now?"
search_results = index.search(
    question,
    boost_dict={"question": 2.0, "section": 0.5},
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

In [43]:
def search(question, course="llm-zoomcamp"):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=3
    )

In [44]:
search_results = search(question)

In [45]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

In [46]:
USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

In [47]:
def build_context(search_results):
    lines = []
    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

In [48]:
def build_prompt(question,search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
    question = question,
    context = context
    )
    return prompt.strip()

In [49]:
prompt = build_prompt(question, search_results)

In [50]:
prompt

'Question:\nI just discovered the course. Can I join now?\n\nContext:\nGeneral Course-Related Questions\nQ: I just discovered the course. Can I still join?\nA: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.\n\nGeneral Course-Related Questions\nQ: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?\nA: You don\'t need it. You\'re accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.\n\nGeneral Course-Related Questions\nQ: Certificate: Can I follow the course in a self-paced mode and get a certificate?\nA: No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after sub

In [53]:
message_history = [
    {"role": "developer", "content": INSTRUCTIONS},
    {"role": "user", "content": prompt}
]

response = openai_client.responses.create(
    model="gpt-5.4-nano",
    input=message_history
)

In [54]:
input_price = 0.2 / 1_000_000
output_price = 1.25 / 1_000_000

cost = (
    response.usage.input_tokens * input_price +
    response.usage.output_tokens * output_price
)

In [55]:
cost

0.00010000000000000002

In [56]:
def llm(instructions, user_prompt, model="gpt-5.4-nano"):
    message_history = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.create(
        model=model,
        input=message_history
    )

    return response.output_text

In [57]:
def rag(query, model="gpt-5.4-nano"):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer

In [58]:
answer = rag("I just discovered the course. Can I join now?")
print(answer)

Yes, you can still join. If you want to receive a certificate, you’ll need to submit your project while we’re still accepting submissions.


In [60]:
from dotenv import load_dotenv
load_dotenv()

from ingest import load_faq_data, build_index
from rag_helper import RAGBase
from openai import OpenAI

documents = load_faq_data()


In [61]:
index = build_index(documents)

openai_client = OpenAI()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
)

answer = assistant.rag("I just discovered the course. Can I join now?")
print(answer)

Yes, you can still join.  
If you want to receive a certificate, you’ll need to submit your project while we’re still accepting submissions.


In [62]:
custom_instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=custom_instructions,
)

In [63]:
assistant.rag("How do I get a certificate?")


'To get a certificate, you must **finish the course with a “live” cohort**.\n\n- **Self-paced mode:** You **can’t** get a certificate. Certificates aren’t awarded in self-paced because you need to **peer-review 3 capstone(s)** after submitting your project.\n- **When peer-review happens:** You can only peer-review while the course is running (after the peer-review list is compiled, you can’t peer-review anymore).\n- **Also:** You need to **pass the Capstone project** to get the certificate (homework is not mandatory).'

In [3]:
from elasticsearch import Elasticsearch
from ingest import load_faq_data, build_elasticsearch_index
from rag_helper import RAGBase
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()
docs = load_faq_data()
client = Elasticsearch("http://localhost:9200")

In [4]:
index = build_elasticsearch_index(docs, client, recreate=True)
print(index.search("Can I still join the course after it started?", num_results=3))

assistant = RAGBase(index=index, llm_client=OpenAI())
print(assistant.rag("Can I still join the course after it started?"))

[{'id': '74eb249bbf', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'I just discovered the course. Can I still join?', 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}, {'id': '9f689c185f', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'I missed the first homework - can I still get a certificate?', 'answer': 'Yes, you need to pass the Capstone project to get the certificate. Homework is not mandatory, though it is recommended for reinforcing concepts, and the points awarded count towards your rank on the leaderboard.'}, {'id': 'aa310de435', 'course': 'llm-zoomcamp', 'section': 'Module 1: RAG', 'question': 'Can I run the course locally instead of Codespaces?', 'answer': 'Yes. Codespaces is just the easiest way for everyone to start with the same environment.\n\nYou can run the course locally if you are comfortable setting up Py

In [5]:
client.indices.delete(index="faq")

ObjectApiResponse({'acknowledged': True})

In [6]:
##AGENTS PART 2

In [20]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [21]:
from rag_helper import RAGBase
from elasticsearch import Elasticsearch
from ingest import load_faq_data, build_elasticsearch_index

documents = load_faq_data()
client = Elasticsearch("http://localhost:9200")
index = build_elasticsearch_index(documents, client, recreate=True)

In [22]:
instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

In [23]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.responses.create(
    model="gpt-5.4-nano",
    input=messages,
)

response.output_text

'Yes—usually you can join as long as enrollment is still open.\n\nTo tell you exactly, I need one detail: **what course/platform is it on (e.g., Coursera, Udemy, a university LMS, etc.)**, and **what message you’re seeing** when you try to join (if any).  \n\nIn the meantime, here are the common cases:\n- **Open enrollment:** You can join immediately (usually you’ll see an “Enroll/Join” button).\n- **Limited seats / cohort-based:** You may need to wait for the next start date.\n- **Instructor approval required:** You may need to request access.\n- **Closed / past start date:** Sometimes you can still enroll but with restricted access, or you can’t enroll at all.\n\nTell me the platform/course name (or paste the join page text), and I’ll guide you step-by-step.'

In [24]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [25]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [26]:
response = openai_client.responses.create(
    model="gpt-5.4-nano",
    input=messages,
    tools=[search_tool],
)


In [28]:
import json

call = response.output[0]
args = json.loads(call.arguments)

results = search(**args)
result_json = json.dumps(results, indent=2)

In [29]:
messages.extend(response.output)

messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
})

In [37]:
response = openai_client.responses.create(
    model="gpt-5.4-nano",
    input=messages,
    tools=[search_tool],
)

response.output_text

'Yes—you can still join. If you want to receive a certificate, make sure you submit your project while the course is still accepting submissions.'

In [38]:
def calculate_gpt54nano_price(input_tokens, output_tokens):
    INPUT_PRICE_PER_MILLION = 0.2
    OUTPUT_PRICE_PER_MILLION = 1.25

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

result = calculate_gpt54nano_price(652, 33)
print("Total cost: $", round(result["total_cost"], 8))

Total cost: $ 0.00017165


In [39]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [18]:
TOOLS = [
    {   "type" : "function",
        "name": "search",
        "description": "Search the FAQ database for entries matching the given query.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "Search query text to look up in the course FAQ."
                }
            },
            "required": ["query"],
            "additionalProperties": False
        }
    }
]

In [40]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [42]:
question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

response = openai_client.responses.create(
    model="gpt-5.4-nano",
    input=messages,
    tools=[search_tool],
)

messages.extend(response.output)
has_function_calls = False

for item in response.output:
    if item.type == "function_call":
        print("function_call:", item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)
        has_function_calls = True

    elif item.type == "message":
        print("ASSISTANT:")
        print(item.content[0].text)

function_call: search {"query":"join course discovered just discovered can I join it enrollment start date"}
function_call: search {"query":"course join late enrollment add to course after start policy"}
function_call: search {"query":"FAQ how to enroll in the course still open"}


In [44]:
it = 1

while True:
    print(f"iteration #{it}...")
    has_function_calls = False

    response = openai_client.responses.create(
        model="gpt-5.4-nano",
        input=messages,
        tools=[search_tool],
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == "function_call":
            print("function_call:", item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == "message":
            print("ASSISTANT:")
            print(item.content[0].text)

    it = it + 1
    if has_function_calls == False:
        break

iteration #1...
ASSISTANT:
Yes—you can still join the course.

You can start learning whenever you want (the videos/GitHub materials are available). If you want to receive a **certificate**, make sure you **submit your project while submissions are still open**.

Are you joining for the certificate or just to follow along self-paced?


In [47]:
def agent_loop(instructions, question, model="gpt-5.4-nano") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break

    return last_answer

In [46]:
agent_loop(instructions, "How do I run Olama locally?")

iteration #1...
function_call: search {"query":"Olama local run install locally Ollama course FAQ"}
iteration #2...
function_call: search {"query":"Ollama serve local server localhost 11434 run llama3 Python client ollama FAQ"}
iteration #3...
ASSISTANT:
To run Ollama locally:

1. **Install Ollama**
   - **macOS**: download the `.pkg` from [ollama.com/download](https://ollama.com/download)
   - **Windows**: download the `.msi`
   - **Linux**:
     ```bash
     curl -fsSL https://ollama.com/install.sh | sh
     ```

2. **Start a model locally**
   Open a terminal and run:
   ```bash
   ollama run llama3
   ```
   This will download the model and start a local chat session.

3. **Check that the local server is running**
   ```bash
   curl http://localhost:11434
   ```
   You should get a response showing the server is up.

4. **Use it from Python**
   ```bash
   pip install ollama
   ```

   Example:
   ```python
   import ollama

   response = ollama.chat(
       model='llama3',
       

'To run Ollama locally:\n\n1. **Install Ollama**\n   - **macOS**: download the `.pkg` from [ollama.com/download](https://ollama.com/download)\n   - **Windows**: download the `.msi`\n   - **Linux**:\n     ```bash\n     curl -fsSL https://ollama.com/install.sh | sh\n     ```\n\n2. **Start a model locally**\n   Open a terminal and run:\n   ```bash\n   ollama run llama3\n   ```\n   This will download the model and start a local chat session.\n\n3. **Check that the local server is running**\n   ```bash\n   curl http://localhost:11434\n   ```\n   You should get a response showing the server is up.\n\n4. **Use it from Python**\n   ```bash\n   pip install ollama\n   ```\n\n   Example:\n   ```python\n   import ollama\n\n   response = ollama.chat(\n       model=\'llama3\',\n       messages=[{"role": "user", "content": "Hello!"}]\n   )\n\n   print(response[\'message\'][\'content\'])\n   ```\n\nIf you get a connection refused error, restart the server with:\n```bash\nollama serve\n```\n\nIf you wa

In [48]:
agent_loop(instructions, "I just discovered the course. Can I still join it?")


iteration #1...
function_call: search {"query":"join course still discover just discovered can I still join deadline add course after start"}
function_call: search {"query":"enrollment late join policy start date late enrollment still join"}
function_call: search {"query":"course FAQ late registration join after start"}
iteration #2...
ASSISTANT:
Yes—you can still join even if you just discovered the course.

If you want to **receive a certificate**, you’ll need to **submit the Capstone project before submissions close**. (Homework is not required for the certificate; passing the Capstone is what matters.)

Do you want help finding where to submit the Capstone / check the submission deadline?


'Yes—you can still join even if you just discovered the course.\n\nIf you want to **receive a certificate**, you’ll need to **submit the Capstone project before submissions close**. (Homework is not required for the certificate; passing the Capstone is what matters.)\n\nDo you want help finding where to submit the Capstone / check the submission deadline?'

In [49]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "I just discovered the course. Can I join it?")

iteration #1...
function_call: search {"query":"just discovered the course can I join it join course late registration"}
iteration #2...
ASSISTANT:
Yes—you can still join. Since you just discovered the course, you can start learning right away.

If you want to receive a **certificate**, you’ll need to **submit your project while we’re still accepting submissions** (the deadline matters for certificate eligibility).

If you tell me whether you’re mainly aiming for a certificate or just self-study, I can point you to the right next step. Are there any other areas you want to explore (e.g., deadlines, how submissions work, or how to start the modules)?


'Yes—you can still join. Since you just discovered the course, you can start learning right away.\n\nIf you want to receive a **certificate**, you’ll need to **submit your project while we’re still accepting submissions** (the deadline matters for certificate eligibility).\n\nIf you tell me whether you’re mainly aiming for a certificate or just self-study, I can point you to the right next step. Are there any other areas you want to explore (e.g., deadlines, how submissions work, or how to start the modules)?'

In [50]:
agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"Queen gambit what is it"}
iteration #2...
ASSISTANT:
A **Queen’s Gambit** is a **chess opening** (a specific sequence of moves), not something related to the LLM/RAG topics in the course.

- **Core idea:** White plays the pawn from **c4** and “offers” it (the gambit), usually aiming to control the **center** and develop pieces actively.
- **Most famous line:** It often starts like:  
  **1. d4 d5 2. c4** (this is the gambit offer)  
  Then Black can accept (…dxc4) or decline.

If you meant the **Netflix show “The Queen’s Gambit”** (about chess and a fictional character), tell me and I’ll summarize that instead.

Are you asking about the **chess opening** or the **TV series**?


'A **Queen’s Gambit** is a **chess opening** (a specific sequence of moves), not something related to the LLM/RAG topics in the course.\n\n- **Core idea:** White plays the pawn from **c4** and “offers” it (the gambit), usually aiming to control the **center** and develop pieces actively.\n- **Most famous line:** It often starts like:  \n  **1. d4 d5 2. c4** (this is the gambit offer)  \n  Then Black can accept (…dxc4) or decline.\n\nIf you meant the **Netflix show “The Queen’s Gambit”** (about chess and a fictional character), tell me and I’ll summarize that instead.\n\nAre you asking about the **chess opening** or the **TV series**?'

In [ ]:
def search(query: str, num_results: int = 5, boost_dict: dict[str, float] | None = None, filter_dict: dict[str, Any] | None = None,
           multi_match_type: str = "best_fields",) -> list[dict[str, Any]]:
    
    boost_dict = boost_dict or {"question": 3.0, "answer": 2.0, "section": 0.5}
    fields = [f"{field}^{boost}" if boost != 1 else field for field, boost in boost_dict.items()]

    query_body: dict[str, Any] = {
        "bool": {
            "must": {
                "multi_match": {
                    "query": query,
                    "fields": fields,
                    "type": multi_match_type,
                }
            },
            "filter": build_filters(filter_dict),
        }
    }

    response = self.client.search(
        index=self.index_name,
        size=num_results,
        query=query_body,
    )
    return [hit["_source"] for hit in response["hits"]["hits"]]

In [16]:
question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

response = openai_client.responses.create(
    model="gpt-5.4-nano",
    input=messages,
    tools=[search_tool],
)

messages.extend(response.output)
has_function_calls = False

for item in response.output:
    if item.type == "function_call":
        print("function_call:", item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)
        has_function_calls = True

    elif item.type == "message":
        print("ASSISTANT:")
        print(item.content[0].text)

NameError: name 'search_tool' is not defined